<a id="unit-u.a02acbb2cb40417194c6fafcfd5c9776"></a>

<div>
  <div>1-ЗЕРТХАНАЛЫҚ ЖҰМЫСТЫҢ ҚОСЫМША БӨЛІМІ · APACHE DORIS КӨМЕГІМЕН НАҚТЫ УАҚЫТТАҒЫ АНАЛИТИКА</div>
  <div>1-зертханалық жұмыстың қосымша бөлімі: Metabase қосу және бақылау тақтасын жасау</div>
  <p>Metabase қызметін 1-зертханалық жұмыстың Doris сынақ ортасының жанында іске қосыңыз, Apache Doris қауымдастық драйверін орнатыңыз, <code>events</code> кестесіне қосылып, күнделікті түсім графигін бақылау тақтасына жариялаңыз.</p>
  <span>Қосымша бөлім · Metabase 0.59.6.3 · Metabase Apache Doris Driver 1.0.0 · Docker · Бақылау тақтасы</span>
</div>

Бұл қосымша бөлім міндетті емес және шамамен 20–30 минут алады. Ол Doris жүйесінің BI қолданбасына арналған талдамалық дерекқор ретінде қалай жұмыс істейтінін көрсетеді; 1-зертханалық жұмысты аяқтау үшін оны орындау міндетті емес.


<a id="unit-u.b9b24fecf63a427c955bf971f88680d9"></a>

<a id="initialize-the-lab-1-optional"></a>

### 1-зертханалық жұмыстың қосымша бөлімін бастапқы баптау

Жалғастырмас бұрын келесі ұяшықты бір рет орындаңыз. Ол 1-зертханалық жұмыста қолданылған жергілікті безендіру мен пәрмен орындаушысын жүктейді. Бағдарламаларды жүктемейді, контейнерлерді іске қоспайды және Doris деректерін өзгертпейді. Jupyter ядросын қайта іске қосқаннан кейін оны қайта орындаңыз.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(
    lab_dir=COURSE_ROOT,
    ready_title="Lab 1 Optional tools are ready",
    ready_message="Continue to Section 1 and run the remaining cells in order.",
);


<a id="unit-u.6b061846acbb4371bce5879f7db52923"></a>

<a id="1-understand-the-integration"></a>

## 1. Интеграцияны түсіну

Metabase — бастапқы коды ашық бизнес-аналитика қолданбасы. Metabase Apache Doris Driver — Metabase жүйесіне Doris метадеректерін табуға және MySQL үйлесімді протоколы арқылы FE `9030` портына SQL жіберуге мүмкіндік беретін дербес қауымдастық драйвері.

| Компонент | Осы қосымша зертханалық жұмыстағы рөлі |
|---|---|
| Браузер | Хосттың `13000` портында Metabase веб-интерфейсін ашады. |
| Metabase контейнері | Сұрақтар мен бақылау тақталарын сақтайды, Doris драйверін жүктейді және SQL жібереді. |
| `doris-rta` Docker желісі | `doris-rta` хост атауы арқылы Doris контейнерінің мекенжайын шешеді. |
| Frontend (FE) | `9030` портында SQL қабылдап, талдайды, үлестірілген сұрау жоспарын құрып, оңтайландырады және оның жоспар фрагменттерін тағайындайды. |
| Backend (BE) | 1-зертханалық жұмыстың `events` кестесін сканерлеу және агрегаттау үшін тағайындалған жоспар фрагменттерін орындайды. |

Metabase контейнері `doris-rta-metabase-data` атаулы томын пайдаланады, сондықтан оның қолданба дерекқоры контейнер тоқтатылғаннан кейін сақталады. Doris кестесі 1-зертханалық жұмыста жасалған бөлек `doris-rta-fe-meta` және `doris-rta-be-storage` томдарында қалады.

<div><strong>Үшінші тарап драйвері</strong>Metabase Apache Doris Driver Apache Doris жобасынан тыс қолдау табады. Ноутбук 1.0.0 нұсқасын орнатады. Өндірісте пайдаланбас бұрын оның қауіпсіздігін, лицензиясын және үйлесімділігін бөлек бағалаңыз.</div>


<a id="unit-u.470819e0e0dc49ec87d4d7a075766fef"></a>

<a id="2-prepare-the-metabase-environment"></a>

## 2. Metabase ортасын дайындау

Алдымен 1-зертханалық жұмысты аяқтаңыз. Келесі ұяшыққа дұрыс жұмыс істейтін `doris-rta` контейнері, `doris-rta` Docker желісі және 10,158,080 жолы бар `doris_course.events` кестесі қажет.

<div>
<table>
<tr>
<td><strong>1 · Doris тексеру</strong><br><span>Контейнерді, желіні және бастапқы кестені тексеру</span></td>
<td>&#8594;</td>
<td><strong>2 · Драйверді жүктеу</strong><br><span>Нұсқасы бекітілген қауымдастық драйверінің JAR файлын алу</span></td>
<td>&#8594;</td>
<td><strong>3 · Сақтау орнын жасау</strong><br><span>Metabase атаулы томын жасау немесе қайта пайдалану</span></td>
<td>&#8594;</td>
<td><strong>4 · Metabase іске қосу</strong><br><span>Образды қайта пайдалану немесе жүктеу және контейнерді іске қосу немесе қайта пайдалану</span></td>
<td>&#8594;</td>
<td><strong>5 · Дайындықты күту</strong><br><span>Metabase күйін тексеру API тексеру</span></td>
</tr>
</table>
</div>

Bash ұяшығының басындағы айнымалылар барлық қайта пайдаланылатын ресурс атаулары мен нұсқаларын басқарады. Сол пәрмендер macOS Apple Silicon және Linux x86_64 жүйелерінде орындалады; Docker платформаға сәйкес образ архитектурасын таңдайды.

In [ ]:
lab.shell(r"""
set -euo pipefail

METABASE_IMAGE="metabase/metabase:v0.59.6.3"
METABASE_CONTAINER="doris-rta-metabase"
METABASE_VOLUME="doris-rta-metabase-data"
METABASE_PORT="${DORIS_RTA_METABASE_PORT:-13000}"  # host port for the Metabase web interface
DORIS_CONTAINER="doris-rta"
DORIS_NETWORK="doris-rta"
DRIVER_VERSION="v1.0.0"
DRIVER_FILE="doris.metabase-driver-v1.0.0.jar"
DRIVER_URL="https://github.com/velodb/metabase-doris-driver/releases/download/${DRIVER_VERSION}/${DRIVER_FILE}"

LAB_DIR="$PWD"
if [ ! -f "$LAB_DIR/pyproject.toml" ]; then
  echo "Start JupyterLab from the 01-real-time-analytics course directory."
  exit 2
fi
ASSETS_DIR="$LAB_DIR/level1/module01-introduction/assets"
DRIVER_PATH="$ASSETS_DIR/$DRIVER_FILE"

echo "[1/5] Verify the Lab 1 Doris sandbox and baseline table"
if ! command -v docker >/dev/null 2>&1; then
  echo "Docker CLI was not found. Complete Lab 1 before running this optional lab."
  exit 3
fi
if ! docker info >/dev/null 2>&1; then
  echo "Docker is not ready. Start Docker Desktop or Docker Engine, then rerun this cell."
  exit 4
fi
if ! docker container inspect "$DORIS_CONTAINER" >/dev/null 2>&1; then
  echo "The Lab 1 Doris container does not exist. Complete Lab 1 first."
  exit 5
fi
if [ "$(docker inspect --format '{{.State.Status}}' "$DORIS_CONTAINER")" != "running" ]; then
  docker start "$DORIS_CONTAINER"
fi
for attempt in $(seq 1 150); do
  DORIS_HEALTH=$(docker inspect --format '{{if .State.Health}}{{.State.Health.Status}}{{else}}none{{end}}' "$DORIS_CONTAINER")
  [ "$DORIS_HEALTH" = "healthy" ] && break
  [ "$attempt" -eq 150 ] && { echo "Doris did not become healthy."; exit 6; }
  sleep 2
done
if ! docker network inspect "$DORIS_NETWORK" >/dev/null 2>&1; then
  echo "The $DORIS_NETWORK network is missing. Rerun the Lab 1 environment preparation."
  exit 7
fi
EVENT_ROWS=$(docker exec "$DORIS_CONTAINER" mysql -uroot -h127.0.0.1 -P9030 -N \
  -e "SELECT COUNT(*) FROM doris_course.events")
if [ "$EVENT_ROWS" != "10158080" ]; then
  echo "Expected 10158080 rows in doris_course.events; found $EVENT_ROWS."
  exit 8
fi

echo "[2/5] Download the pinned Apache Doris community driver"
mkdir -p "$ASSETS_DIR"
if [ ! -f "$DRIVER_PATH" ]; then
  curl -fL --retry 3 --output "$DRIVER_PATH" "$DRIVER_URL"
else
  echo "Reusing $DRIVER_PATH"
fi

echo "[3/5] Create or reuse persistent Metabase storage"
docker volume inspect "$METABASE_VOLUME" >/dev/null 2>&1 || docker volume create "$METABASE_VOLUME"

echo "[4/5] Reuse or pull the pinned image and start or reuse Metabase"
docker image inspect "$METABASE_IMAGE" >/dev/null 2>&1 || docker pull "$METABASE_IMAGE"
if docker container inspect "$METABASE_CONTAINER" >/dev/null 2>&1; then
  CURRENT_IMAGE=$(docker inspect --format '{{.Config.Image}}' "$METABASE_CONTAINER")
  CURRENT_NETWORKS=$(docker inspect --format '{{range $name, $_ := .NetworkSettings.Networks}}{{println $name}}{{end}}' "$METABASE_CONTAINER")
  CURRENT_MOUNTS=$(docker inspect --format '{{range .Mounts}}{{printf "%s|%s\n" .Name .Destination}}{{end}}' "$METABASE_CONTAINER")
  # Read the configured bindings; `docker port` shows nothing while the container is stopped.
  CURRENT_PORTS=$(docker inspect --format '{{range $port, $bindings := .HostConfig.PortBindings}}{{range $bindings}}{{printf "%s %s:%s\n" $port .HostIp .HostPort}}{{end}}{{end}}' "$METABASE_CONTAINER")
  CURRENT_ENV=$(docker inspect --format '{{range .Config.Env}}{{println .}}{{end}}' "$METABASE_CONTAINER")
  if [ "$CURRENT_IMAGE" != "$METABASE_IMAGE" ] \
     || ! grep -Fxq "$DORIS_NETWORK" <<<"$CURRENT_NETWORKS" \
     || ! grep -Fxq "$METABASE_VOLUME|/metabase-data" <<<"$CURRENT_MOUNTS" \
     || ! grep -Fq '|/plugins/doris.metabase-driver.jar' <<<"$CURRENT_MOUNTS" \
     || { [ -n "${DORIS_RTA_METABASE_PUBLIC_URL:-}" ] && ! grep -Fxq "MB_SITE_URL=${DORIS_RTA_METABASE_PUBLIC_URL}" <<<"$CURRENT_ENV"; } \
     || ! grep -Fxq "3000/tcp 127.0.0.1:${METABASE_PORT}" <<<"$CURRENT_PORTS"; then
    echo "The existing $METABASE_CONTAINER container does not match this optional lab configuration."
    echo "It was left unchanged; rename it or choose different resource names above."
    exit 11
  fi
  if [ "$(docker inspect --format '{{.State.Status}}' "$METABASE_CONTAINER")" != "running" ]; then
    docker start "$METABASE_CONTAINER"
  fi
else
  if (exec 3<>"/dev/tcp/127.0.0.1/${METABASE_PORT}") 2>/dev/null; then
    echo "Host port ${METABASE_PORT} is already in use. Stop the conflicting program, or set"
    echo "DORIS_RTA_METABASE_PORT to a free port before starting Jupyter."
    exit 12
  fi
  docker run -d \
    --name "$METABASE_CONTAINER" \
    --network "$DORIS_NETWORK" \
    -p "127.0.0.1:${METABASE_PORT}:3000" \
    -e MB_DB_FILE=/metabase-data/metabase.db \
    -e "MB_SITE_URL=${DORIS_RTA_METABASE_PUBLIC_URL:-http://127.0.0.1:$METABASE_PORT}" \
    -v "$METABASE_VOLUME:/metabase-data" \
    -v "$DRIVER_PATH:/plugins/doris.metabase-driver.jar:ro" \
    "$METABASE_IMAGE"
fi

echo "[5/5] Wait for the Metabase health API"
for attempt in $(seq 1 150); do
  if HEALTH_RESPONSE=$(curl -fsS --max-time 3 "http://127.0.0.1:${METABASE_PORT}/api/health" 2>/dev/null); then
    echo "$HEALTH_RESPONSE"
    break
  fi
  if [ "$attempt" -eq 150 ]; then
    docker logs --tail 100 "$METABASE_CONTAINER"
    echo "Metabase did not become ready within 5 minutes."
    exit 13
  fi
  [ $((attempt % 10)) -eq 0 ] && echo "Waiting for Metabase (${attempt}/150)..."
  sleep 2
done
docker ps --filter "name=^/${METABASE_CONTAINER}$"
""", title="Prepare the Metabase environment");

<a id="unit-u.1534db6795b94ba496d4c880ddb8bfd2"></a>

**Күтілетін нәтиже:** жұмыс барысының барлық бес тармағы сәтті аяқталады. Күй жауабында `"status":"ok"` бар, ал `docker ps` пәрмені `doris-rta-metabase` және `doris-rta` контейнерлерінің екеуі де жұмыс істеп тұрғанын көрсетеді. Жүктелген драйвер `assets/` ішінде қалады, ал Metabase қолданбасының деректері `doris-rta-metabase-data` атаулы томында сақталады.

Ұяшық сәтсіз аяқталса, **View diagnostic log** ашыңыз. Порт қақтығысы жаңа Metabase контейнері жасалғанға дейін жұмыс барысын тоқтатады.

<a id="unit-u.d50106693c794aac81a8247118ad29ee"></a>

<a id="3-open-metabase-and-add-doris"></a>

## 3. Metabase ашып, Doris қосу

Төмендегі Metabase браузер сілтемесін ашыңыз. Алғаш іске қосқанда Metabase әкімшісін баптауды аяқтаңыз. Әкімші тіркелгісі Metabase жүйесіне тиесілі; ол Doris пайдаланушысынан бөлек.

Metabase 0.59.6.3 нұсқасында жоғарғы оң жақтағы **Settings** мәзірін ашыңыз, содан кейін **Admin → Databases → Add database** тармағын таңдаңыз, **Apache Doris** таңдап, мыналарды енгізіңіз:

| Өріс | Мән |
|---|---|
| Display name | `Doris Course` |
| Host | `doris-rta` |
| Port | `9030` |
| Catalog | `internal` |
| Database | `doris_course` |
| Username | `root` |
| Password | Тек осы уақытша жергілікті сынақ ортасында бос қалдырыңыз |
| SSL | Өшірілген |

> **Database өрісіне `doris_course` орнатыңыз:** Metabase бұл өрісті міндетті емес деп көрсеткенімен, оны бос қалдырсаңыз, `internal` Catalog ішіндегі бірнеше дерекқорды іздеу қосылады. Сонда Metabase бірнеше көрінетін дерекқордың кестелерін көрсете алады және толық нақтыланбаған `events` атауы `internal.doris_course.events` кестесін сенімді түрде білдірмейді.

Host ретінде `127.0.0.1` орнына `doris-rta` пайдаланыңыз. Metabase контейнерінің ішінде `127.0.0.1` Metabase жүйесінің өзін білдіреді; Docker DNS ортақ `doris-rta` желісіндегі `doris-rta` атауын Doris контейнерінің мекенжайына шешеді.

Қосылым Database көрсетілмей сақталған болса, Metabase контейнерін немесе метадерек томын жоймаңыз. **Settings → Admin → Databases → Doris Course** ашып, қосылымды өңдеңіз, **Database** өрісін `doris_course` деп орнатып, сақтаңыз. Содан кейін **Sync database schema now** орындаңыз. Бұл түзету жасалғанша, Native SQL ішінде қажетті кестеге `internal.doris_course.events` толық атауымен жүгінуге болады.

**Күтілетін нәтиже:** қосылым сынағы сәтті өтіп, синхрондау `internal.doris_course` дерекқорымен шектеледі. Metabase курстың `events` кестесін табады; қосылым сақталғаннан кейін метадеректерді синхрондау аз уақыт жалғасуы мүмкін.


In [ ]:
from doris_course import environment as course_environment
lab.show_browser_link(f"http://127.0.0.1:{course_environment.METABASE_PORT}", "Open Metabase");

<a id="unit-u.632cdec224a142609e1f562e3a13b6b8"></a>

**Күтілетін нәтиже:** браузерде ашуға арналған бір сілтеме пайда болады. Оны ашып, жұмыс істеп тұрған қызметке өтіңіз. Бұл зертханалық жұмыс қашықтағы хостта орындалса, сілтеме бапталған браузер мекенжайын пайдаланады; хост жағындағы API сұраулары ішкі мекенжайын сақтайды.

<a id="unit-u.fa1cd8e7d0b641db8b275ffbfd9fc540"></a>

<a id="4-create-a-question-and-line-chart"></a>

## 4. Сұрақ пен сызықтық график жасау

**New → SQL query** таңдаңыз, **Doris Course** таңдап, орындаңыз:

```sql
SELECT
    TO_DATE(event_time) AS event_date,
    SUM(revenue) AS total_revenue
FROM internal.doris_course.events
WHERE event_type = 'purchase'
GROUP BY TO_DATE(event_time)
ORDER BY event_date;
```

Catalog, дерекқор және кестеден тұратын толық кесте атауы Native SQL ұқсас аталған `information_schema.EVENTS` жүйелік көрінісінің орнына курс кестесін оқитынын қамтамасыз етеді. `TO_DATE(event_time)` бір күнтізбелік күндегі барлық сатып алуды топтайды.

**Visualization → Line** таңдаңыз, X осіне `event_date`, Y осіне `total_revenue` орнатыңыз және Question сұрағын `Our analytics` Collection жинағына **Daily Revenue** атымен сақтаңыз.

**Күтілетін нәтиже:** сұрау хронологиялық ретпен жеті жол қайтарады, сатып алу болған әр күнге бір жол: 2019-12-01 мен 2019-12-04 аралығы және 2020-03-01 мен 2020-03-03 аралығы. Деректер жиынының қалған 19 күнінде қарау немесе себет оқиғалары бар, бірақ сатып алу жоқ, сондықтан түсім де жоқ. Сызықтық график қосымша деректер қатарысыз күнделікті сатып алу түсімін көрсетеді: 2019-12-04 күнгі 2,756,102.03 мәнінен 2020-03-02 күнгі 9,933,097.98 мәніне дейін.
Графиктегі жеті нүкте ғана сұрау нәтижелері болып табылады. Әдепкі сызық қаңтар мен ақпанды аттап өтіп, 2019-12-04 күнін 2020-03-01 күнімен байланыстырады; бұл бөлік аралықтағы күнделікті бақылауларды білдірмейді. Сатып алу күні туралы деректің болмауы мен өлшенген нөл есеп берудегі әртүрлі таңдаулар болғандықтан, бұл сұрау жетіспейтін күндерді нөлмен толтырмайды.


<a id="unit-u.68f3ba3ec19441058c69ddb50e69d68c"></a>

<a id="5-publish-the-dashboard"></a>

## 5. Бақылау тақтасын жариялау

Жасау диалогы бос Dashboard тақтасын және Metabase оны қайда сақтайтынын анықтайды. Диалогта көрсетілген өрістерді толтырыңыз:

1. **+ New → Dashboard** таңдаңыз.
2. **Name** өрісіне `Doris Course Overview` енгізіңіз.
3. **Description** өрісіне `Daily purchase revenue from Apache Doris.` енгізіңіз.
4. **Which collection should this go in?** астынан `Our analytics` таңдаңыз.
5. **Create** таңдаңыз.

Metabase жаңа бос Dashboard тақтасын өңдеу режимінде ашады. Осы экраннан сақталған Question сұрағын қосыңыз:

6. Dashboard тақтасының жоғарғы оң жағындағы **Add questions** деп белгіленген **+** түймесін таңдаңыз.
7. Ашылған бүйірлік панельден сақталған `Daily Revenue` Question сұрағын тауып, таңдаңыз.
8. Сызықтық график карточкасын Dashboard тақтасына орналастырыңыз, қажет болса өлшемін өзгертіңіз.
9. Dashboard орналасуын сақтау үшін жоғарғы оң жақтағы **Save** таңдаңыз.

**Күтілетін нәтиже:** `Doris Course Overview` тақтасы `Our analytics` ішінде сақталады және `Daily Revenue` атты бір карточкадан тұрады. Карточка күнделікті сатып алу түсімінің сызықтық графигін көрсетеді. Dashboard тақтасынан шығып, оны қайта ашқанда карточка мен оның орналасуы сақталады.


<a id="unit-u.865aac9394df4a0da038260482bfe237"></a>

<a id="stop-metabase"></a>

### Metabase тоқтату

Metabase үдерісін және 13000 портын босатқыңыз келгенде осы қосымша ұяшықты орындаңыз. Ол контейнер мен `doris-rta-metabase-data` атаулы томын сақтайды, сондықтан Doris қосылымы, сақталған Question сұрағы және бақылау тақтасы жойылмайды. Doris контейнері мен `events` кестесіне әсер етпейді.


In [ ]:
lab.shell(r"""
set -euo pipefail

docker stop doris-rta-metabase
docker inspect --format 'container={{.State.Status}}' doris-rta-metabase
""", title="Stop Metabase");


<a id="unit-u.6b6eb00732674ec2a7dc4b2c5c570c20"></a>

**Күтілетін нәтиже:** Docker `container=exited` деп хабарлайды; Metabase атаулы томы қолжетімді болып қалады.

<a id="restart-metabase"></a>

### Metabase қызметін қайта іске қосу

Сақталған бақылау тақтасын қайта ашқыңыз келгенде осы ұяшықты орындаңыз. Ол Doris тәуелділігін қалпына келтіреді, бар Metabase контейнерін іске қосады немесе қайта пайдаланады және Metabase күйін тексеру API қолданбаның дайын екенін хабарлағанша күтеді.

Бұл қайта іске қосу ұяшығы идемпотентті: оны Docker Desktop пен контейнер тоқтаған, іске қосылып жатқан немесе жұмыс істеп тұрған кезде орындауға болады. macOS жүйесінде қажет болса Docker Desktop ашылады; Linux жүйесінде ұяшықты орындамас бұрын Docker Engine іске қосыңыз.


In [ ]:
lab.start_container()
lab.start_container("doris-rta-metabase", wait_for_healthy=False)

lab.shell(r"""
set -euo pipefail

for attempt in $(seq 1 150); do
  if HEALTH_RESPONSE=$(curl -fsS --max-time 3 "http://127.0.0.1:${DORIS_RTA_METABASE_PORT:-13000}/api/health" 2>/dev/null); then
    echo "$HEALTH_RESPONSE"
    break
  fi

  if [ "$attempt" -eq 150 ]; then
    docker logs --tail 100 doris-rta-metabase
    exit 1
  fi

  if [ $((attempt % 10)) -eq 0 ]; then
    echo "Waiting for Metabase (${attempt}/150)..."
  fi
  sleep 2
done
""", title="Restart Metabase");

<a id="unit-u.2bceba5e236a4056a9380faaf4c39bdc"></a>

**Күтілетін нәтиже:** күй жауабында `"status":"ok"` бар. Төмендегі Metabase браузер сілтемесін қайта ашыңыз; Doris қосылымы, сақталған Question сұрағы және бақылау тақтасы қолжетімді болып қалады.

<a id="common-problems"></a>

### Жиі кездесетін мәселелер

| Белгісі | Тексеру |
|---|---|
| Дерекқор түрлері тізімінде **Apache Doris** жоқ | `docker logs doris-rta-metabase` орындап, `/plugins/doris.metabase-driver.jar` тіркелгенін растаңыз. |
| Қосылым сынағы сәтсіз аяқталды | Host мәні `doris-rta`, Port мәні `9030` екенін, екі контейнер де `doris-rta` желісін пайдаланатынын және Doris контейнерінің дұрыс жұмыс істейтінін растаңыз. |
| `events` көрінбейді | Database мәні `doris_course` екенін растаңыз, содан кейін Metabase ішінде **Sync database schema now** орындаңыз. |
| `DESC events` нәтижесінде `EVENT_CATALOG` және `EVENT_SCHEMA` көрсетіледі | Толық нақтыланбаған атау `information_schema.EVENTS` ретінде шешілді. Native SQL ішінде `internal.doris_course.events` пайдаланыңыз. |
| Браузер 13000 портын аша алмайды | `docker ps`, күйді тексеру мекенжайын және хосттағы басқа үдерістің `13000` портын иеленіп тұрмағанын тексеріңіз. |

<a id="lab-1-optional-complete"></a>

## 1-зертханалық жұмыстың қосымша бөлімі аяқталды

Сіз қауымдастық драйверін орнаттыңыз, Metabase пен Doris жүйелерін бір Docker желісінде іске қостыңыз, Metabase қызметін Doris FE түйініне қостыңыз, Doris ішінде талдамалық сұрау орындадыңыз және нәтижесін сақталатын бақылау тақтасы ретінде жарияладыңыз.

Ресми анықтама: [Apache Doris — Metabase интеграциясы](https://doris.apache.org/docs/4.x/connection-integration/data-integration/metabase/)


In [ ]:
from doris_course import environment as course_environment
lab.show_browser_link(f"http://127.0.0.1:{course_environment.METABASE_PORT}", "Open Metabase");

<a id="unit-u.8ef1c70a1d404b53bbddd364f9816d84"></a>

**Күтілетін нәтиже:** браузерде ашуға арналған бір сілтеме пайда болады. Оны ашып, жұмыс істеп тұрған қызметке өтіңіз. Бұл зертханалық жұмыс қашықтағы хостта орындалса, сілтеме бапталған браузер мекенжайын пайдаланады; хост жағындағы API сұраулары ішкі мекенжайын сақтайды.